# Multi-Disease Lung Pathology Detection from Chest X-Rays
## Using Deep Learning and Explainable AI
### Notebook 5 — Grad-CAM Explainability (XAI)

**Dataset:** CheXpert v1.0-small — Stanford University ML Group
**Best model:** VGG16 (selected by Notebook 4, Mean AUC = 0.7492)
**Reference:** Selvaraju et al. (2020). *Grad-CAM.* IJCV, 128, 336–359.

---

| Section | Content |
|---------|---------|
| 1 | Environment Setup |
| 2 | Load Best Model |
| 3 | Grad-CAM Implementation |
| 4 | Grad-CAM++ Implementation |
| 5 | Image Utilities & Test Split |
| 6 | Single-Image Explanation |
| 7 | All 14 Pathologies Panel |
| 8 | Gallery — One Image per Pathology |
| 9 | Grad-CAM vs Grad-CAM++ Comparison |
| 10 | Pointing Game — Quantitative Evaluation |
| 11 | Summary |

---

## 1. Environment Setup

In [ ]:
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
plt.rcParams['font.family']      = 'DejaVu Sans'

# ── Paths ────────────────────────────────────────────────────────────────
BASE_PATH = '/kaggle/input/datasets/ashery/chexpert'
OUT_DIR   = '/kaggle/working'
NB4_JSON  = '/kaggle/input/notebooks/parashqeviklimi/notebook-4-model-comparison-best-model-selecti/nb4_best_model.json'
VGG16_H5  = '/kaggle/input/notebooks/parashqeviklimi/notebook-3c-transfer-learning-with-vgg16/vgg16_best.h5'

IMG_SIZE  = 224

LABEL_COLS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices'
]
NUM_CLASSES = len(LABEL_COLS)

print('TensorFlow :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))
print('BASE_PATH  :', os.path.exists(BASE_PATH))
print('NB4_JSON   :', os.path.exists(NB4_JSON))
print('VGG16_H5   :', os.path.exists(VGG16_H5))


---
## 2. Load Best Model

Loads VGG16 checkpoint saved by NB3C. Reads NB4 selection JSON for reference.

In [ ]:
# Load NB4 selection info (for reference / reporting)
with open(NB4_JSON) as f:
    selection = json.load(f)
BEST_MODEL = selection['best_model']   # 'vgg16'
print('Best model  :', BEST_MODEL.upper())
print('Mean AUC    :', round(selection['best_auc_test'], 4))
print('Loading from:', VGG16_H5)

# Load model — compile=False, inference only
model = tf.keras.models.load_model(VGG16_H5, compile=False)
model.trainable = False
print('Model loaded. Layers:', len(model.layers))
model.summary(line_length=80, show_trainable=False)


---
## 3. Grad-CAM Implementation

**Grad-CAM** (Selvaraju et al., 2020) weights the final convolutional feature maps by the gradient of the target class score, then applies ReLU.

$$\alpha_k^c = \frac{1}{Z}\sum_i\sum_j \frac{\partial y^c}{\partial A_{ij}^k} \qquad L^c = \text{ReLU}\!\left(\sum_k \alpha_k^c A^k\right)$$

In [ ]:
def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError('No Conv2D layer found.')

LAST_CONV = get_last_conv_layer(model)
print('Last conv layer:', LAST_CONV)

def compute_gradcam(model, img_array, class_idx, last_conv_name):
    grad_model = Model(
        inputs  = model.input,
        outputs = [model.get_layer(last_conv_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)
        tape.watch(inputs)
        conv_outputs, predictions = grad_model(inputs, training=False)
        class_score = predictions[:, class_idx]
    grads = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_outputs[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.nn.relu(heatmap).numpy()
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()
    return heatmap, float(predictions[0, class_idx])

print('compute_gradcam defined.')


---
## 4. Grad-CAM++ Implementation

**Grad-CAM++** (Chattopadhay et al., 2018) uses second-order gradients for better spatial coverage.

In [ ]:
def compute_gradcampp(model, img_array, class_idx, last_conv_name):
    grad_model = Model(
        inputs  = model.input,
        outputs = [model.get_layer(last_conv_name).output, model.output]
    )
    with tf.GradientTape() as t2:
        with tf.GradientTape() as t1:
            with tf.GradientTape() as t0:
                inputs = tf.cast(img_array, tf.float32)
                t0.watch(inputs); t1.watch(inputs); t2.watch(inputs)
                conv_out, preds = grad_model(inputs, training=False)
                score = preds[:, class_idx]
            g1 = t0.gradient(score, conv_out)
        g2 = t1.gradient(g1, conv_out)
    g3 = t2.gradient(g2, conv_out)
    g1 = g1.numpy(); g2 = g2.numpy(); g3 = g3.numpy()
    denom = 2 * g2[0] + conv_out.numpy()[0] * g3[0]
    denom = np.where(denom != 0, denom, 1e-10)
    alpha = g2[0] / denom
    alpha = np.where(g1[0] > 0, alpha, 0)
    weights = np.sum(alpha * np.maximum(g1[0], 0), axis=(0, 1))
    heatmap = np.sum(weights * conv_out.numpy()[0], axis=-1)
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return heatmap, float(preds[0, class_idx])

print('compute_gradcampp defined.')


---
## 5. Image Utilities & Test Split

Helper functions for overlay and preprocessing. Rebuilds the exact same test split as NB03 (SEED=42).

In [ ]:
def overlay_heatmap(orig_img, heatmap, alpha=0.45, colormap=cv2.COLORMAP_JET):
    """Superimpose Grad-CAM heatmap on original image."""
    h, w = orig_img.shape[:2]
    hm_resized = cv2.resize(heatmap, (w, h))
    hm_uint8   = np.uint8(255 * hm_resized)
    hm_color   = cv2.applyColorMap(hm_uint8, colormap)
    hm_rgb     = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB)
    overlay    = cv2.addWeighted(orig_img, 1 - alpha, hm_rgb, alpha, 0)
    return overlay

def load_and_preprocess(path):
    """Load X-ray -> (1,224,224,3) float32 tensor + (224,224,3) uint8 display."""
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=1)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.repeat(img, 3, axis=-1)
    img_f     = tf.cast(img, tf.float32) / 255.0
    img_batch = tf.expand_dims(img_f, 0).numpy()
    img_disp  = np.uint8(img_f.numpy() * 255)
    return img_batch, img_disp

# ── Rebuild test split identical to NB03 ─────────────────────────────────
U_ONES = ['Edema', 'Atelectasis', 'Pleural Effusion', 'Cardiomegaly']

train_raw = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
frontal   = train_raw[train_raw['Frontal/Lateral'] == 'Frontal'].copy()

for col in LABEL_COLS:
    fill = 1.0 if col in U_ONES else 0.0
    frontal[col] = frontal[col].fillna(0.0).replace(-1.0, fill).astype(np.float32)

frontal['full_path'] = frontal['Path'].str.replace(
    'CheXpert-v1.0-small/', '', regex=False
).apply(lambda p: os.path.join(BASE_PATH, p))

_, tmp = train_test_split(frontal, test_size=0.20, random_state=SEED,
                          stratify=frontal['Pleural Effusion'])
_, test_df = train_test_split(tmp, test_size=0.50, random_state=SEED,
                              stratify=tmp['Pleural Effusion'])
test_df = test_df.sample(n=5000, random_state=SEED).reset_index(drop=True)

print('Test split  :', len(test_df), 'samples')
print('Path exists :', os.path.exists(test_df['full_path'].iloc[0]))


---
## 6. Single-Image Explanation

Full Grad-CAM + Grad-CAM++ panel for one Pleural Effusion positive sample.

In [ ]:
sample   = test_df[test_df['Pleural Effusion'] == 1.0].iloc[0]
path     = sample['full_path']
gt       = {c: int(sample[c]) for c in LABEL_COLS}

img_batch, img_disp = load_and_preprocess(path)
preds = model.predict(img_batch, verbose=0)[0]
top3_idx = np.argsort(preds)[::-1][:3]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(f'Figure 6 — Grad-CAM & Grad-CAM++ | VGG16 | {os.path.basename(path)}',
             fontsize=12, fontweight='bold')

axes[0, 0].imshow(img_disp); axes[0, 0].set_title('Original X-Ray', fontweight='bold'); axes[0, 0].axis('off')
axes[1, 0].axis('off')
pred_text = 'Top predictions:\n' + '\n'.join(
    f'  {LABEL_COLS[i]}: {preds[i]:.3f}' for i in top3_idx)
axes[1, 0].text(0.05, 0.9, pred_text, transform=axes[1,0].transAxes,
                fontsize=9, va='top', family='monospace')

plot_items = [
    (compute_gradcam,   'Grad-CAM',        cv2.COLORMAP_JET),
    (compute_gradcampp, 'Grad-CAM++',      cv2.COLORMAP_JET),
    (compute_gradcam,   'Grad-CAM (plasma)',cv2.COLORMAP_PLASMA),
]
for col_offset, (fn, name, cmap) in enumerate(plot_items):
    for row, class_idx in enumerate(top3_idx[:2]):
        heatmap, score = fn(model, img_batch, class_idx, LAST_CONV)
        overlaid = overlay_heatmap(img_disp, heatmap, colormap=cmap)
        ax = axes[row, col_offset + 1]
        ax.imshow(overlaid)
        ax.set_title(f'{name}\n{LABEL_COLS[class_idx]} ({score:.3f})', fontsize=9, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_nb5_single_explanation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_nb5_single_explanation.png')


---
## 7. All 14 Pathologies Panel

One Grad-CAM overlay per label on the same image.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(25, 16))
fig.suptitle('Figure 7 — Grad-CAM for All 14 Pathologies | VGG16',
             fontsize=14, fontweight='bold')

axes[0, 0].imshow(img_disp)
axes[0, 0].set_title('Original', fontweight='bold', fontsize=10)
axes[0, 0].axis('off')

for idx, col in enumerate(LABEL_COLS):
    ax = axes[(idx + 1) // 5, (idx + 1) % 5]
    heatmap, score = compute_gradcam(model, img_batch, idx, LAST_CONV)
    overlaid = overlay_heatmap(img_disp, heatmap)
    gt_flag  = chr(10003) if gt.get(col, 0) == 1 else chr(10007)
    ax.imshow(overlaid)
    ax.set_title(f'{col}\nP={score:.3f} GT={gt_flag}', fontsize=7, fontweight='bold')
    ax.axis('off')

axes[2, 4].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_nb5_all14_panel.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_nb5_all14_panel.png')


---
## 8. Gallery — One Representative Image per Pathology

Finds one positive sample per label and generates its Grad-CAM.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(25, 16))
fig.suptitle('Figure 8 — Grad-CAM Gallery: One Representative per Pathology | VGG16',
             fontsize=13, fontweight='bold')

for idx, col in enumerate(LABEL_COLS):
    ax = axes[idx // 5, idx % 5]
    positives = test_df[test_df[col] == 1.0]
    if len(positives) == 0:
        ax.axis('off'); ax.set_title(f'{col}\n(no positives)', fontsize=8)
        continue
    row_s = positives.sample(1, random_state=SEED + idx).iloc[0]
    img_b, img_d = load_and_preprocess(row_s['full_path'])
    heatmap, score = compute_gradcam(model, img_b, idx, LAST_CONV)
    ax.imshow(overlay_heatmap(img_d, heatmap))
    ax.set_title(f'{col}\nP={score:.3f}', fontsize=8, fontweight='bold')
    ax.axis('off')

axes[2, 4].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_nb5_gallery.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_nb5_gallery.png')


---
## 9. Grad-CAM vs Grad-CAM++ — Key Pathologies

Side-by-side comparison for the 4 clinically critical labels (U-Ones group).

In [ ]:
KEY_LABELS = ['Cardiomegaly', 'Edema', 'Atelectasis', 'Pleural Effusion']

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
fig.suptitle('Figure 9 — Grad-CAM vs Grad-CAM++ | Key Pathologies | VGG16',
             fontsize=13, fontweight='bold')

for row, col in enumerate(KEY_LABELS):
    class_idx = LABEL_COLS.index(col)
    positives = test_df[test_df[col] == 1.0]
    s = positives.sample(1, random_state=SEED + row).iloc[0]
    img_b, img_d = load_and_preprocess(s['full_path'])
    hm_gc,   sc_gc   = compute_gradcam(model,   img_b, class_idx, LAST_CONV)
    hm_gcpp, sc_gcpp = compute_gradcampp(model, img_b, class_idx, LAST_CONV)
    axes[row, 0].imshow(img_d)
    axes[row, 0].set_title(f'Original — {col}', fontweight='bold', fontsize=9)
    axes[row, 0].axis('off')
    axes[row, 1].imshow(overlay_heatmap(img_d, hm_gc))
    axes[row, 1].set_title(f'Grad-CAM (P={sc_gc:.3f})', fontsize=9)
    axes[row, 1].axis('off')
    axes[row, 2].imshow(overlay_heatmap(img_d, hm_gcpp))
    axes[row, 2].set_title(f'Grad-CAM++ (P={sc_gcpp:.3f})', fontsize=9)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_nb5_gradcam_vs_pp.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_nb5_gradcam_vs_pp.png')


---
## 10. Pointing Game — Quantitative Evaluation

Measures whether Grad-CAM attends more strongly on **positive** samples than **negative** ones.
A positive delta (Δ = pos_activation − neg_activation) confirms the model attends to pathology-relevant regions.

In [ ]:
N_EVAL = 200
eval_results = {}

print(f'{'Pathology':<35} {'Pos act':>10} {'Neg act':>10} {'Delta':>8}')
print('-' * 68)

for idx, col in enumerate(LABEL_COLS):
    pos_df = test_df[test_df[col] == 1.0].sample(
        min(N_EVAL, (test_df[col] == 1.0).sum()), random_state=SEED)
    neg_df = test_df[test_df[col] == 0.0].sample(
        min(N_EVAL, (test_df[col] == 0.0).sum()), random_state=SEED)

    def mean_max(df_sub):
        acts = []
        for _, r in df_sub.iterrows():
            try:
                ib, _ = load_and_preprocess(r['full_path'])
                hm, _ = compute_gradcam(model, ib, idx, LAST_CONV)
                acts.append(float(hm.max()))
            except Exception:
                pass
        return float(np.mean(acts)) if acts else 0.0

    p = mean_max(pos_df)
    n = mean_max(neg_df)
    d = p - n
    eval_results[col] = {'pos': p, 'neg': n, 'delta': d}
    print(f'{col:<35} {p:>10.4f} {n:>10.4f} {d:>8.4f}')

print('-' * 68)
mean_delta = np.mean([v['delta'] for v in eval_results.values()])
print(f'{'MEAN DELTA':<35} {mean_delta:>29.4f}')

with open(os.path.join(OUT_DIR, 'nb5_pointing_game.json'), 'w') as f:
    json.dump(eval_results, f, indent=2)
print('Saved: nb5_pointing_game.json')


In [ ]:
labels_  = list(eval_results.keys())
deltas   = [eval_results[c]['delta'] for c in labels_]
colors   = ['#388E3C' if d > 0 else '#E53935' for d in deltas]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(labels_, deltas, color=colors, alpha=0.85)
ax.axhline(0, color='black', lw=1)
ax.set_xticks(range(len(labels_)))
ax.set_xticklabels(labels_, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Delta Max Activation (Positive - Negative)')
ax.set_title('Figure 10 — Pointing Game: Grad-CAM Localisation Quality | VGG16',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_nb5_pointing_game.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_nb5_pointing_game.png')


---
## 11. Summary

| Figure | Content | File |
|--------|---------|------|
| 6 | Single-image Grad-CAM + Grad-CAM++ panel | `fig_nb5_single_explanation.png` |
| 7 | All 14 pathologies on one image | `fig_nb5_all14_panel.png` |
| 8 | Gallery: one representative per pathology | `fig_nb5_gallery.png` |
| 9 | Grad-CAM vs Grad-CAM++ for key pathologies | `fig_nb5_gradcam_vs_pp.png` |
| 10 | Pointing Game quantitative evaluation | `fig_nb5_pointing_game.png` |

---

## References

1. Selvaraju, R. R., et al. (2020). **Grad-CAM.** *IJCV*, 128, 336–359.
2. Chattopadhay, A., et al. (2018). **Grad-CAM++.** *WACV*, 839–847.
3. Zhang, J., et al. (2018). **Excitation Backprop.** *IJCV*, 126, 1084–1102.
4. Irvin, J., et al. (2019). **CheXpert.** *AAAI*, 33(01), 590–597.